In [2]:
# Setup: run this cell, then build the analysis below it.
from pathlib import Path
from urllib.request import urlopen
import json

import pandas as pd
import numpy as np

RELATIVE_DATA_PATH = Path("data/week02_python_objects_posts.json")
RAW_DATA_URL = (
    "https://raw.githubusercontent.com/macss-berkeley/compss-211a/"
    "main/data/week02_python_objects_posts.json"
)

def find_local_file(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    return None

local_data = find_local_file(RELATIVE_DATA_PATH)
if local_data is not None:
    data_source = str(local_data)
    records = json.loads(local_data.read_text(encoding="utf-8"))
else:
    data_source = RAW_DATA_URL
    with urlopen(RAW_DATA_URL, timeout=30) as response:
        records = json.load(response)

posts = pd.DataFrame(records)
display(posts.head())
print(f"Loaded {len(posts)} rows from {data_source}")

,post_id,author,platform,text,likes,verified,region,year,publication_status
0,1,user_101,forum,Public transit should be free.,27.0,False,West,2024,published
1,2,user_102,forum,The city needs more protected bike lanes.,42.0,True,West,2024,published
2,3,user_103,microblog,None,5.0,False,South,2025,published
3,4,user_104,microblog,Remote work has changed downtown neighborhoods.,31.0,True,Northeast,2025,published
4,5,user_105,forum,Libraries are essential public infrastructure.,18.0,False,South,2025,published


Loaded 90 rows from /Users/Angie/compss-211a/data/week02_python_objects_posts.json


### Your manager's request

"We can focus our next transit outreach pilot on **one platform**. Which should we prioritize, and why?"

Work with the same partner throughout. Use the **text to identify relevant posts**, then compare their **likes** across platforms.

Test whether a minimum-like rule helps remove irrelevant material, and inspect which useful posts it would exclude. Likes are a screening signal to investigate; read the text to judge relevance.

By the end of class, produce **one summary table or chart and a recommendation**. Explain your time period, any like cutoff, what counted as relevant, and one uncertainty that could change your advice.


### Duo task 1 · Decide which posts to include

Choose **both years** or **2025 only**, the latest year in the data. Explain what your choice gains and leaves out.

Inspect some posts with low like counts. Would a minimum-like rule help remove irrelevant material? **Try different thresholds**, such as 10 and 20 likes. For each, identify an irrelevant post it excludes and a relevant post it excludes. Use the text to judge relevance.

Choose a like cutoff, or keep posts regardless of likes. Apply your period and like-count rules to `published_posts` and save the result as `analysis_posts`. Keep all columns, use `.copy()`, and apply the same rules to both platforms. Keep `published_posts` available to inspect exclusions later.

*Hint: combine conditions using `&`, with parentheses around each condition.*

Check what happens at exactly your threshold and when the like count is missing.

**Discuss:** What irrelevant material does your rule remove? What useful transit discussion does it lose? Does that tradeoff help answer the manager's question?

In [8]:
posts.shape

(90, 9)

In [ ]:
posts.columns

In [ ]:
posts['publication_status'] == "published"

In [ ]:
published = posts[posts['publication_status'] == "published"]
published = published[published['likes'] >= 20]


In [ ]:
is_forum = published['platform'] == "forum"
enough_likes = published['likes'] >= 20
keep = is_forum & enough_likes
example_posts = published[keep]
example_posts

In [17]:
published_base = posts[posts['publication_status'] == "published"]

In [18]:
published_base['likes'].mean()

np.float64(38.166666666666664)

In [ ]:
published_base['likes'].describe()

count     78.000000
mean      38.166667
std       46.848586
min        0.000000
25%       12.000000
50%       23.000000
75%       47.000000
max      300.000000
Name: likes, dtype: float64

In [22]:
published_base['year'].value_counts()

year
2025    56
2024    28
Name: count, dtype: int64

In [23]:
published_2025 = published_base[published_base['year'] == 2025]

In [24]:
published_2024 = published_base[published_base['year'] == 2024]

In [25]:
published_2025.shape

(56, 9)

In [ ]:
published_2024.shape # looks like there are many fewer in 2024 > 2025

(28, 9)

In [28]:
published_2024['likes'].describe()

count     27.000000
mean      26.592593
std       26.533374
min        0.000000
25%        9.000000
50%       12.000000
75%       41.500000
max      100.000000
Name: likes, dtype: float64

In [29]:
published_2025['likes'].describe()

count     51.000000
mean      44.294118
std       53.881831
min        0.000000
25%       18.000000
50%       24.000000
75%       49.000000
max      300.000000
Name: likes, dtype: float64

In [32]:
published_less_than_10 = published_base[published_base['likes'] < 10]
published_less_than_10['likes'].describe()

count    14.000000
mean      3.857143
std       3.324898
min       0.000000
25%       0.000000
50%       4.500000
75%       6.750000
max       8.000000
Name: likes, dtype: float64

In [33]:
published_less_than_20 = published_base[published_base['likes'] < 20]
published_less_than_20['likes'].describe()

count    33.000000
mean     10.000000
std       6.269968
min       0.000000
25%       5.000000
50%      10.000000
75%      15.000000
max      19.000000
Name: likes, dtype: float64

In [34]:
analysis_posts = published_less_than_20

In [ ]:
published_less_than_20

### Duo task 2 · Decide what counts as relevant

Choose a keyword rule for transit outreach. Should it cover bus services, bike lanes, or commuting?

Write `classify_topic(text)`, using the demo as a starting point. Return `match` when your rule matches, `no_match` when it does not, and `unknown` when text is missing or blank.

Test a clear match, a non-match, a borderline example, and absent text. Apply the function to create `analysis_posts["topic_status"]`. 

Create a new DF `topic_posts` that holds all the relevant posts.

**Discuss:** Which relevant posts might your rule miss, and which might it include by mistake? Revisit your excluded examples in `published_posts`: did the like cutoff remove useful discussion under your topic definition?


In [ ]:
in_2025 = published['year'] == 2025
analysis_posts = published[in_2025]
analysis_posts.head()

,post_id,author,platform,text,likes,verified,region,year,publication_status
3,4,user_104,microblog,Remote work has changed downtown neighborhoods.,31.0,True,Northeast,2025,published
8,9,user_101,forum,Free transit would help people reach evening s...,20.0,False,West,2025,published
10,11,user_102,forum,"Make transit free, but fund reliable service too.",21.0,True,West,2025,published
11,12,user_110,forum,The last bus leaves before my shift ends.,35.0,True,South,2025,published
13,14,user_111,forum,Housing near transit is still unaffordable.,44.0,False,Northeast,2025,published


In [42]:
keywords = ["metro", "transit", "bus", "train", "subway", "light rail", "streetcar", "trolley", "cable car", "monorail", "tram"]
def transit_keyword(text):
    if pd.isna(text):
        return "unknown"
    text = text.strip().lower()
    if text == "":
        return "unknown"
    if "transit" in text:
        return "match"
    if "metro" in text:
        return "match"
    if "bus" in text:
        return "match"
    if "train" in text:
        return "match"
    if "subway" in text:
        return "match"
    if "light rail" in text:
        return "match"
    if "streetcar" in text:
        return "match"
    if "trolley" in text:
        return "match"
    if "cable car" in text:
        return "match"
    if "monorail" in text:
        return "match"
    if "tram" in text:
        return "match"
    return "no match"



In [45]:
analysis_posts['transit_keyword'] = analysis_posts['text'].apply(transit_keyword)
analysis_posts['text']

/var/folders/kj/hdn95bp52wd_mjs7hvhmtds00000gp/T/ipykernel_80834/184984594.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  analysis_posts['transit_keyword'] = analysis_posts['text'].apply(transit_keyword)


3       Remote work has changed downtown neighborhoods.
8     Free transit would help people reach evening s...
10    Make transit free, but fund reliable service too.
11            The last bus leaves before my shift ends.
13          Housing near transit is still unaffordable.
15    Bike lanes need physical protection, not just ...
16    Public meetings should offer remote participat...
19         Our neighborhood park has no working lights.
21    Which streets should get protected bike lanes ...
23    We need more buses before we advertise free ri...
24    A new bus route would connect the clinic and l...
28                  My commute now takes two transfers.
29    Accessible sidewalks are part of public transp...
30     Free transit sounds good. How will it be funded?
34    I support free transit for students and low-in...
40                                Free transit, please.
46              A shorter commute would change my week.
47                                              

### Duo task 3 · Give the manager advice

Use `groupby` on `topic_posts` to compare platforms. Show the **number of matching posts and known like counts** beside the result.

Choose **mean or median likes** to guide your recommendation. Explain what that measure tells the manager.

Check how one alternative changes the recommendation: your other like cutoff, the period, topic rule, or summary measure. Change one choice at a time. If you change the period or cutoff, rebuild `analysis_posts` from `published_posts`. After changing an inclusion or topic rule, rerun `.apply()`, rebuild `topic_posts`, and recalculate.

Finish with one table or chart labeled with your **period, like-count rule, and topic rule**. Recommend a platform and explain one uncertainty. Use an actual excluded post to explain what your selection leaves out; account for missing text or likes separately.
